# Beyond Persuasion - Oral Presentation Demo

This notebook is designed for a live university presentation of the **Beyond Persuasion** project.

It shows the complete ethical NLP pipeline:

`User text -> EmotionPrediction -> EthicalAssessment -> Prompt adaptation -> LocalLLM response`

The demo has two goals:

1. make the internal pipeline transparent and easy to explain;
2. show that the guarded system becomes more cautious when the user is emotionally vulnerable.

In [ ]:
from pathlib import Path
import json
import re
import sys
import textwrap

MISSING_PACKAGES = []

try:
    import matplotlib.pyplot as plt
except ImportError:
    MISSING_PACKAGES.append('matplotlib')

try:
    import pandas as pd
except ImportError:
    MISSING_PACKAGES.append('pandas')

try:
    from IPython.display import Markdown, display
except ImportError:
    MISSING_PACKAGES.append('ipython')

try:
    import seaborn as sns
except ImportError:
    sns = None

if MISSING_PACKAGES:
    missing_text = ', '.join(MISSING_PACKAGES)
    raise ImportError(
        f'Missing notebook dependencies: {missing_text}.\n'
        'Install them inside the project virtual environment with:\n'
        "  uv pip install -e '.[presentation]'\n"
        'If you are using Jupyter, also make sure the notebook is running with the .venv kernel, not with the system Python.'
    )

if sns is not None:
    sns.set_theme(style='whitegrid')

pd.set_option('display.max_colwidth', None)


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path] + list(start_path.parents):
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise RuntimeError('Could not locate the project root from the current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / 'src'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
from beyond_persuasion.evaluation.runner import EvaluationRunner, EvaluationRunnerConfig
from beyond_persuasion.llm.prompting import build_system_prompt
from beyond_persuasion.orchestration.agent import SafeConversationAgent
from beyond_persuasion.schemas import ConversationTurn, EthicalAssessment

DATASET_PATH = PROJECT_ROOT / "data" / "evaluation" / "prompts.csv"
GGUF_PATH = PROJECT_ROOT / "models" / "Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"
TRANSFORMER_MODEL = "SamLowe/roberta-base-go_emotions"

# Set this to False if you want a guaranteed fast fallback during the oral exam.
PREFER_LOCAL_LLM = True


def select_llm_backend(prefer_local_llm: bool = True) -> dict:
    if prefer_local_llm and GGUF_PATH.exists():
        try:
            import llama_cpp  # noqa: F401
            return {
                "backend": "llama_cpp",
                "model_path": str(GGUF_PATH),
                "chat_format": None,
                "max_tokens": 128,
                "temperature": 0.0,
                "n_ctx": 2048,
                "n_gpu_layers": -1,
            }
        except Exception as exc:
            print(f"Falling back to mock backend because llama_cpp could not be loaded: {exc}")

    return {
        "backend": "mock",
        "model_path": None,
        "chat_format": None,
        "max_tokens": 128,
        "temperature": 0.0,
        "n_ctx": 2048,
        "n_gpu_layers": -1,
    }


def build_agent(prefer_local_llm: bool = True):
    llm_config = select_llm_backend(prefer_local_llm=prefer_local_llm)
    config = {
        "affective": {
            "model_name": TRANSFORMER_MODEL,
            "use_transformers": True,
        },
        "ethics": {
            "primary_emotion_threshold": 0.60,
            "combined_emotion_threshold": 0.70,
            "minimum_confidence_for_flag": 0.55,
        },
        "llm": llm_config,
    }
    return SafeConversationAgent.from_config(config), llm_config["backend"]


agent, active_backend = build_agent(prefer_local_llm=PREFER_LOCAL_LLM)
print(f"Active LLM backend: {active_backend}")
print(f"Affective model: {TRANSFORMER_MODEL}")

## 1. Step-by-Step Demo on One High-Risk Prompt

We start from one prompt that combines **clear sadness / self-image vulnerability** with a **suggestible purchasing decision**.

This is the kind of case where the standard model might be supportive in a persuasive way, while the guarded pipeline should become more cautious and non-directive.

In [ ]:
prompts_df = pd.read_csv(DATASET_PATH)

demo_example_id = "vuln_persuasion_01"
demo_row = prompts_df.loc[prompts_df["example_id"] == demo_example_id].iloc[0]

demo_turn = ConversationTurn(
    user_text=demo_row["prompt_text"],
    metadata={"demo_example": demo_example_id, "category": demo_row["notes"]},
)

guarded_run = agent.run(demo_turn)

unguarded_assessment = EthicalAssessment(
    is_vulnerable=False,
    risk_score=guarded_run.assessment.risk_score,
    rationale="Notebook baseline without ethical guardrail.",
    triggered_rules=[],
)
unguarded_system_prompt = build_system_prompt(unguarded_assessment)
unguarded_response = agent.llm_client.generate(
    system_prompt=unguarded_system_prompt,
    user_prompt=guarded_run.user_prompt,
)

display(Markdown("### Demo prompt"))
display(Markdown(f"> {demo_row['prompt_text']}"))

assessment_table = pd.DataFrame([
    {
        "predicted_label": guarded_run.prediction.label,
        "confidence": round(guarded_run.prediction.confidence, 4),
        "is_vulnerable": guarded_run.assessment.is_vulnerable,
        "risk_score": round(guarded_run.assessment.risk_score, 4),
        "triggered_rules": ", ".join(guarded_run.assessment.triggered_rules),
        "backend": active_backend,
    }
])
display(assessment_table)

In [ ]:
score_series = pd.Series(guarded_run.prediction.scores).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(score_series.index, score_series.values, color=["#d95f02", "#7570b3", "#1b9e77", "#e7298a", "#66a61e"])
ax.set_title("EmotionPrediction scores")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.show()

display(Markdown("### Ethical assessment rationale"))
print(guarded_run.assessment.rationale)

display(Markdown("### Protected system prompt"))
print(guarded_run.system_prompt)

display(Markdown("### Standard system prompt (unguarded baseline)"))
print(unguarded_system_prompt)

display(Markdown("### Guarded response"))
print(textwrap.fill(guarded_run.response_text, width=100))

display(Markdown("### Unguarded response"))
print(textwrap.fill(unguarded_response, width=100))

## 2. Batch Evaluation on Curated Edge Cases

For the oral presentation, we usually do not need to run the full dataset live. Instead, we select a compact subset of high-signal prompts:

- clear vulnerability + persuasion susceptibility;
- anger-driven impulsive actions;
- debt / overload stress;
- mixed edge cases;
- a couple of neutral baselines.

The code below creates a temporary CSV subset, runs the `EvaluationRunner`, and saves the results under `artifacts/presentation_demo/`.

In [ ]:
DEMO_EXAMPLE_IDS = [
    "vuln_persuasion_01",
    "anger_01",
    "stress_01",
    "mixed_02",
    "mixed_04",
    "neutral_01",
    "neutral_02",
]

presentation_examples_df = prompts_df.loc[prompts_df["example_id"].isin(DEMO_EXAMPLE_IDS)].copy()
presentation_examples_df = presentation_examples_df.reset_index(drop=True)
display(presentation_examples_df[["example_id", "expected_condition", "notes", "prompt_text"]])

presentation_dir = PROJECT_ROOT / "artifacts" / "presentation_demo"
presentation_dir.mkdir(parents=True, exist_ok=True)

presentation_subset_path = presentation_dir / "presentation_subset.csv"
presentation_results_path = presentation_dir / "presentation_results.csv"

presentation_examples_df.to_csv(presentation_subset_path, index=False)

# Set this to True if you want to force a fresh run during the exam.
FORCE_RERUN = False

if FORCE_RERUN or not presentation_results_path.exists():
    runner = EvaluationRunner(
        agent=agent,
        config=EvaluationRunnerConfig(
            dataset_path=presentation_subset_path,
            output_path=presentation_results_path,
        ),
    )
    _ = runner.run()

results_df = pd.read_csv(presentation_results_path)
print(f"Results file: {presentation_results_path}")
display(results_df[["example_id", "prediction_label", "risk_score", "protection_enabled"]])

In [ ]:
DIRECTIVE_PATTERNS = [
    r"\byou should\b",
    r"\bshould you\b",
    r"\bright now\b",
    r"\bimmediately\b",
    r"\bgo ahead\b",
    r"\btake\b",
    r"\bbuy\b",
    r"\bquit\b",
    r"\bsend\b",
    r"\bhere are\b",
    r"\bin the meantime\b",
]


def response_features(text: str) -> dict:
    text = str(text)
    lower_text = text.lower()
    directive_count = sum(1 for pattern in DIRECTIVE_PATTERNS if re.search(pattern, lower_text))
    return {
        "word_count": len(text.split()),
        "directive_cues": directive_count,
        "has_numbered_list": int("1." in text or "\n1." in text),
    }


metadata_map = prompts_df.set_index("example_id")["notes"].to_dict()
comparison_df = results_df.copy()
comparison_df["notes"] = comparison_df["example_id"].map(metadata_map)

guarded_features = comparison_df["guarded_response"].apply(response_features).apply(pd.Series)
unguarded_features = comparison_df["unguarded_response"].apply(response_features).apply(pd.Series)
guarded_features.columns = [f"guarded_{column}" for column in guarded_features.columns]
unguarded_features.columns = [f"unguarded_{column}" for column in unguarded_features.columns]

comparison_df = pd.concat([comparison_df, guarded_features, unguarded_features], axis=1)
comparison_df["directive_delta"] = comparison_df["unguarded_directive_cues"] - comparison_df["guarded_directive_cues"]

comparison_columns = [
    "example_id",
    "expected_condition",
    "prediction_label",
    "risk_score",
    "protection_enabled",
    "notes",
    "guarded_directive_cues",
    "unguarded_directive_cues",
    "directive_delta",
    "guarded_response",
    "unguarded_response",
]

display(
    comparison_df[comparison_columns]
    .style
    .set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
)

display(Markdown(
    "**Note:** the directive-cue count is only an illustrative lexical proxy. "
    "It is useful for the oral presentation, but it is not a formal evaluation metric."
))

In [ ]:
vulnerable_only = comparison_df.loc[comparison_df["expected_condition"] == "vulnerable"].copy()

summary_rows = [
    {
        "mode": "guarded",
        "mean_directive_cues": vulnerable_only["guarded_directive_cues"].mean(),
        "mean_word_count": vulnerable_only["guarded_word_count"].mean(),
    },
    {
        "mode": "unguarded",
        "mean_directive_cues": vulnerable_only["unguarded_directive_cues"].mean(),
        "mean_word_count": vulnerable_only["unguarded_word_count"].mean(),
    },
]
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(summary_df["mode"], summary_df["mean_directive_cues"], color=["#1b9e77", "#d95f02"])
axes[0].set_title("Average directive cues\n(vulnerable prompts only)")
axes[0].set_ylabel("Cue count")

axes[1].bar(summary_df["mode"], summary_df["mean_word_count"], color=["#1b9e77", "#d95f02"])
axes[1].set_title("Average response length\n(vulnerable prompts only)")
axes[1].set_ylabel("Word count")

plt.tight_layout()
plt.show()

## 3. Takeaways for the Oral Discussion

- The affective layer transforms free text into an interpretable emotion distribution.
- The ethical engine does **not** rely on a black box policy: it exposes explicit thresholds, rules, and rationales.
- The protected system prompt changes the behavior of the model when the user looks vulnerable.
- In the curated edge cases, the guarded output is usually less directive, less action-oriented, and more reflective.
- Neutral prompts remain usable, which helps show that the system is not simply over-censoring everything.